In [2]:
from google.colab import drive
from pathlib import Path
import sys
import os

drive.mount('/content/drive', force_remount=True)

PROJECT_ROOT = Path('/content/drive/Othercomputers/My Laptop/asymmetric_trinet_with_saf_based_rejection')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)

print("Working directory:", os.getcwd())

Mounted at /content/drive
Working directory: /content/drive/Othercomputers/My Laptop/asymmetric_trinet_with_saf_based_rejection


In [3]:
from python.src.train.model_trainer import train_model
from pathlib import Path
import torch
import os

PROJECT_ROOT = Path.cwd()

models = ["asymmetric_trinet"]
version = "v2"
seeds = [216]
n_per_class = [2500]
epochs = 35


for m in models:
  for s in seeds:
    for n in n_per_class:

      print(f"\n\nRunning experiment model = {m}, seed = {s}, n_per_class = {n}")
      print("============================================================================\n")

      trained_model = train_model(seed=s,
                  project_root=PROJECT_ROOT,
                  model_name=m,
                  n_per_class=n,
                  spec_version=version,
                  n_epochs=epochs
                  )
      # clean up RAM
      del trained_model
      if torch.cuda.is_available():
            torch.cuda.empty_cache()
      elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            torch.mps.empty_cache()
      print("\nTraining finished")



Running experiment model = asymmetric_trinet, seed = 216, n_per_class = 2500


Validation report found: /content/drive/Othercomputers/My Laptop/asymmetric_trinet_with_saf_based_rejection/reports/validations/validation_seed216_n2500_v2.json
Starting training...

Using device: cuda

Epoch 01 | LR: 0.001000 | Train: 2.1364 (CE 1.8309 / SupCon 3.0549) | Val Loss: 1.4437 | Val Acc: 66.00
Epoch 05 | LR: 0.000968 | Train: 1.2457 (CE 1.0344 / SupCon 2.1127) | Val Loss: 1.2003 | Val Acc: 75.52
Epoch 10 | LR: 0.000846 | Train: 1.1303 (CE 0.9341 / SupCon 1.9622) | Val Loss: 0.8382 | Val Acc: 86.50
Epoch 15 | LR: 0.000655 | Train: 1.0437 (CE 0.8606 / SupCon 1.8316) | Val Loss: 0.7652 | Val Acc: 89.40
Epoch 20 | LR: 0.000433 | Train: 0.9449 (CE 0.7785 / SupCon 1.6645) | Val Loss: 0.8113 | Val Acc: 89.24
Epoch 25 | LR: 0.000225 | Train: 0.8824 (CE 0.7275 / SupCon 1.5486) | Val Loss: 0.8508 | Val Acc: 88.82
Epoch 30 | LR: 0.000071 | Train: 0.8144 (CE 0.6725 / SupCon 1.4190) | Val Loss: 0.8482 | Val

In [ ]:
from python.src.train.osr_trainer import train_osr_model
from python.src.train.osr_hparams import OSRHParams
from pathlib import Path
import torch

PROJECT_ROOT = Path.cwd()

version = "v2"
seeds = [216]
n_per_class = [2500]
epochs = 30

# Initialize hyperparameters (tweak these directly here if needed)
hparams = OSRHParams()

for s in seeds:
    for n in n_per_class:

        print(f"\n\nRunning OSR experiment | seed = {s}, n_per_class = {n}")
        print("============================================================================\n")

        trained_model = train_osr_model(
            seed=s,
            n_per_class=n,
            spec_version=version,
            project_root=PROJECT_ROOT,
            epochs=epochs,
            hparams=hparams
        )

        # clean up RAM
        del trained_model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            torch.mps.empty_cache()

        print("Trainng Complete")



Running OSR experiment | seed = 216, n_per_class = 2500


OsrSAF_TriNet | seed=216 | n=2500
Device              : cuda
Closed-set ckpt     : asymmetric_trinet_seed216_n2500.pt
Codebook fill epochs: 15
Phase 2 (calibrator): until epoch 30
FPR cap (Youden)    : 0.40
Recal interval      : 5

[load_osr_datasets] proxy unknowns: 7000 samples (train 5600 / val 1400)
[load_osr_datasets] test  unknowns: 3000 samples (held out)
[OsrSAF_TriNet] Loaded backbone from /content/drive/Othercomputers/My Laptop/asymmetric_trinet_with_saf_based_rejection/artifacts/checkpoints/asymmetric_trinet_seed216_n2500.pt

[Stage 2.A] Populating codebook over 15 epochs (frozen backbone)

  Fill epoch 1/15 | init=100% | spread=0.0636 | updates/centroid=380.4
  Fill epoch 2/15 | init=100% | spread=0.0731 | updates/centroid=759.7
  Fill epoch 3/15 | init=100% | spread=0.0722 | updates/centroid=1136.7
  Fill epoch 4/15 | init=100% | spread=0.0747 | updates/centroid=1515.2
  Fill epoch 5/15 | init=100% | spread=0.0646

In [ ]:
import torch
import platform
import psutil

# 1. GPU Info
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU found"
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0

# 2. CPU Info
cpu_info = platform.processor()

# 3. RAM Info
ram_total = psutil.virtual_memory().total / (1024**3)

# 4. PyTorch Version
torch_ver = torch.__version__

print(f"--- THESIS HARDWARE SPECS ---")
print(f"GPU: {gpu_name} ({gpu_mem:.2f} GB VRAM)")
print(f"CPU: {cpu_info} (Intel Xeon)")
print(f"System RAM: {ram_total:.2f} GB")
print(f"PyTorch Version: {torch_ver}")
print(f"-----------------------------")
